In [1]:
from CSVtoSQL import CSVFolderToSQL
CSVFolderToSQL("../synthea/output/csv", "medical_data.db").csv_to_sql()

In [ ]:
import pandas as pd
import numpy as np
from SQL_query import SQLQuery
SQL = SQLQuery("medical_data.db")
query = """
WITH LATEST_PREGNANCY AS (
    SELECT
        PATIENT,
        MAX(DATE(START)) AS PREGNANCY_START,
        DATE(MAX(START), '+270 days') AS EST_DELIVERY_DATE
    FROM encounters
    WHERE DESCRIPTION = 'Prenatal initial visit (regime/therapy)'
    GROUP BY PATIENT
),
LATEST_PREG_DIA AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Diastolic Blood Pressure' THEN o.VALUE END) AS Diastolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Diastolic Blood Pressure' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE AND DATE(o.DATE) >= lp.PREGNANCY_START
    GROUP BY o.PATIENT
),
LATEST_PREG_SYS AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Systolic Blood Pressure' THEN o.VALUE END) AS Systolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Systolic Blood Pressure' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE AND DATE(o.DATE) >= lp.PREGNANCY_START
    GROUP BY o.PATIENT
),
LATEST_PREG_OXSat AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Oxygen saturation in Arterial blood' THEN o.VALUE END) AS Oxygen_Saturation
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Oxygen saturation in Arterial blood' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE AND DATE(o.DATE) >= lp.PREGNANCY_START
    GROUP BY o.PATIENT
),
LATEST_PREG_HDL AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Cholesterol in HDL [Mass/volume] in Serum or Plasma' THEN o.VALUE END) AS HDL_Cholesterol
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Cholesterol in HDL [Mass/volume] in Serum or Plasma' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE AND DATE(o.DATE) >= lp.PREGNANCY_START
    GROUP BY o.PATIENT
),
LATEST_PREG_LDL AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Cholesterol in LDL [Mass/volume] in Serum or Plasma by Direct assay' THEN o.VALUE END) AS LDL_Cholesterol
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Cholesterol in LDL [Mass/volume] in Serum or Plasma by Direct assay' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE AND DATE(o.DATE) >= lp.PREGNANCY_START
    GROUP BY o.PATIENT
)
SELECT 
    t1.Id,
    DATE(t1.BIRTHDATE) AS BIRTHDATE,
    DATE(t1.DEATHDATE) AS DEATHDATE,
    t1.RACE,
    t1.ETHNICITY,
    t1.GENDER,
    t1.COUNTY,
    t1.HEALTHCARE_EXPENSES,
    t1.HEALTHCARE_COVERAGE,
    t1.INCOME,
    DATE(t2.PREGNANCY_START) AS PREGNANCY_START,
    DATE(t2.EST_DELIVERY_DATE) AS EST_DELIVERY_DATE,
    DATE(t3.OBSERVATION_DATE) AS OBSERVATION_DATE,
    t3.Diastolic_Blood_Pressure,
    t4.Systolic_Blood_Pressure,
    t5.Oxygen_Saturation,
    t6.HDL_Cholesterol,
    t7.LDL_Cholesterol
    
FROM patients as t1
INNER JOIN LATEST_PREGNANCY as t2 ON t1.Id = t2.PATIENT
LEFT JOIN LATEST_PREG_DIA as t3 ON t1.Id = t3.PATIENT
LEFT JOIN LATEST_PREG_SYS as t4 ON t1.Id = t4.PATIENT
LEFT JOIN LATEST_PREG_OXSat as t5 ON t1.Id = t5.PATIENT
LEFT JOIN LATEST_PREG_HDL as t6 ON t1.Id = t6.PATIENT
LEFT JOIN LATEST_PREG_LDL as t7 ON t1.Id = t7.PATIENT
"""
LAfemMed = SQL.execute_query(query)
LAfemMed['BIRTHDATE'] = pd.to_datetime(LAfemMed['BIRTHDATE'])
LAfemMed['DEATHDATE'] = pd.to_datetime(LAfemMed['DEATHDATE'])
LAfemMed['PREGNANCY_START'] = pd.to_datetime(LAfemMed['PREGNANCY_START'])
LAfemMed['EST_DELIVERY_DATE'] = pd.to_datetime(LAfemMed['EST_DELIVERY_DATE'])
LAfemMed['OBSERVATION_DATE'] = pd.to_datetime(LAfemMed['OBSERVATION_DATE'])
LAfemMed['Days_Pregancy_to_Death'] = (LAfemMed['DEATHDATE'] - LAfemMed['PREGNANCY_START']).dt.days
LAfemMed['Maternal_Mortality'] = np.where(LAfemMed['Days_Pregancy_to_Death'] <= 292, 1, 0)